In [13]:
import pandas as pd

pd.set_option("display.max_rows",30)

human_Percentages_df=pd.read_csv("../human_predictions/output_all.csv")
llm_Percentages_df=pd.read_csv("../llm_predictions/result_vwp50_scene_table.csv")

llm_Percentages_df=llm_Percentages_df.rename(columns={"Item":"stimuliId","Condition":"condition","Object":"obj"})
llm_Percentages_df["stimuliId"] = llm_Percentages_df["stimuliId"].astype(int)

#OBSOLET AFTER UPDATE
#remove strange star symbols that come up in strings (even though we like stars they dont help here, sry Fei //Jonathan ^^)
#llm_Percentages_df["obj"] = (
#    llm_Percentages_df["obj"]
#    .str.replace(r"^[^\w]+", "", regex=True)
#    .str.strip()
#)

#fixing different denotations of condition:
llm_Percentages_df["condition"] = llm_Percentages_df["condition"].replace("Restrictive", "restrictive")
llm_Percentages_df["condition"] = llm_Percentages_df["condition"].replace("Non-restr.", "non-restrictive")
#r=llm_Percentages_df[llm_Percentages_df["obj"]=="cake"]
#r
llm_Percentages_df

,stimuliId,condition,Verb,obj,Is_Target,Surprisal (bits),P_norm,Rank,ΔS (bits)
0,1,restrictive,eat,cake,True,5.961,0.9734,1,6.482
1,1,restrictive,eat,ball,False,11.156,0.0266,2,-6.202
2,1,restrictive,eat,toy car,False,20.215,0.0000,3,-5.087
3,1,restrictive,eat,toy train,False,22.952,0.0000,4,-6.074
4,1,non-restrictive,move,ball,False,4.953,0.9934,1,-6.202
...,...,...,...,...,...,...,...,...,...
395,50,restrictive,lick,rubber band,False,17.832,0.0203,4,1.492
396,50,non-restrictive,pick up,scissors,False,11.491,0.5686,1,-5.245
397,50,non-restrictive,pick up,envelope,False,12.000,0.3994,2,-0.571
398,50,non-restrictive,pick up,stamp,True,15.759,0.0295,3,0.876


In [14]:
#TODO: investigate human percentages for trials where all are zero
human_Percentages_df

,stimuliId,partID,condition,obj,percent
0,7,1,non-restrictive,weighing-scales,0.000000
1,7,1,non-restrictive,jug,0.000000
2,7,1,non-restrictive,mushrooms,0.000000
3,7,1,non-restrictive,knife,0.000000
4,40,1,restrictive,shampoo,0.000000
...,...,...,...,...,...
995,42,5,restrictive,orange,0.134615
996,46,5,restrictive,laddle,0.000000
997,46,5,restrictive,peeler,0.000000
998,46,5,restrictive,pot,0.000000


In [15]:
#first before comparing we need to aggregate. not over all participants of a trial 
# but over all participants with the same contidion (4-restrictive 4 non restrictive) however some may not be present yet...
avg_df_human = (
    human_Percentages_df
    .groupby(["stimuliId", "condition", "obj"])
    .agg(
        avg_percent=("percent", "mean"),
        n_participants=("partID", "nunique")
    )
    .reset_index()
)
avg_df_human.to_csv("01_intermediate_human_aggregation.csv")
avg_df_human

,stimuliId,condition,obj,avg_percent,n_participants
0,1,non-restrictive,ball,0.0,3
1,1,non-restrictive,cake,0.0,3
2,1,non-restrictive,toy-car,0.0,3
3,1,non-restrictive,toy-train,0.0,3
4,1,restrictive,ball,0.0,2
...,...,...,...,...,...
403,50,restrictive,pouch,0.0,1
404,50,restrictive,pump,0.0,1
405,50,restrictive,rubberband,0.0,2
406,50,restrictive,scissors,0.0,2


In [ ]:
#check after avergaing over all participants how many stimuli got only 0 percentages...

In [24]:
#now we want to create a shared dataframe that maps the percentages contains both human and llm prediction per word per condition

#merge on ritgh because llm should be calculated for all

#maybe there still remains some kind of mismatch ? 400 rows (4*50*2 seems how it should be, why are there more )
#inner 378 --> this is most likely because we dont have values for all combinations yet //participant 6,7,8 are missing. 

#so for know we use inner

shared_df=avg_df_human.merge(llm_Percentages_df, on=["stimuliId","condition","obj"],how="inner")


shared_df

,stimuliId,condition,obj,avg_percent,n_participants,Verb,Is_Target,Surprisal (bits),P_norm,Rank,ΔS (bits)
0,1,non-restrictive,ball,0.0,3,move,False,4.953,0.9934,1,-6.202
1,1,non-restrictive,cake,0.0,3,move,True,12.443,0.0055,2,6.482
2,1,restrictive,ball,0.0,2,eat,False,11.156,0.0266,2,-6.202
3,1,restrictive,cake,0.0,2,eat,True,5.961,0.9734,1,6.482
4,2,non-restrictive,chair,0.0,2,try,False,11.308,0.3230,2,-4.301
...,...,...,...,...,...,...,...,...,...,...,...
373,50,non-restrictive,scissors,0.0,2,pick up,False,11.491,0.5686,1,-5.245
374,50,non-restrictive,stamp,0.0,2,pick up,True,15.759,0.0295,3,0.876
375,50,restrictive,envelope,0.0,2,lick,False,12.571,0.7792,1,-0.571
376,50,restrictive,scissors,0.0,2,lick,False,16.736,0.0435,3,-5.245


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")

fig, ax = plt.subplots(figsize=(8, 8))

sns.scatterplot(
    data=shared_df,
    x="avg_percent",
    y="P_norm",
    hue="condition",       
    style="condition",     
    s=80,                 
    alpha=0.7,             
    edgecolor="white",
    linewidth=0.5,
    palette="viridis",
    ax=ax
)

# Diagonale (perfekte Übereinstimmung) einzeichnen
lims = [0, 1]  # anpassen, falls Werte nicht zwischen 0 und 1 liegen
ax.plot(lims, lims, ls="--", color="gray", alpha=0.6, label="perfect agreement")

ax.set_xlabel("Human Probability")
ax.set_ylabel("LLM Probability")
ax.set_title("LLM predictions vs. Human predictions")